# Sistema de recomendaciones de películas

En este notebook se construye un sistema recomendador básico usando el conjunto de datos de MovieLens/TMDB.  
El objetivo es recomendar películas similares a partir de metadatos como género, palabras clave y popularidad.

También se agrega una mejora: un filtro de popularidad usando calificación ponderada, para evitar recomendar películas con muy pocos votos.

In [ ]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)

## 1. Carga de datos

Se cargan los archivos `movies_metadata.csv` y `keywords.csv`.

In [ ]:
movies = pd.read_csv("data/movies_metadata.csv", low_memory=False)
keywords = pd.read_csv("data/keywords.csv")

display(movies.head())
display(keywords.head())

## 2. Limpieza inicial

Algunos identificadores pueden venir con valores no numéricos, por eso se convierten cuidadosamente a número.  
Después se unen los datos de películas con las palabras clave.

In [ ]:
movies = movies[movies["id"].astype(str).str.isnumeric()].copy()
movies["id"] = movies["id"].astype(int)
keywords["id"] = keywords["id"].astype(int)

movies = movies.merge(keywords, on="id", how="left")

movies = movies.drop_duplicates(subset="id")

print("Tamaño del dataset:", movies.shape)
movies[["id", "title", "genres", "keywords", "vote_average", "vote_count"]].head()

## 3. Funciones auxiliares

Las columnas `genres` y `keywords` vienen como texto con estructura tipo lista de diccionarios.  
Con estas funciones extraigo solamente los nombres importantes.

In [ ]:
def parse_names(value):
    try:
        items = ast.literal_eval(value)
        if isinstance(items, list):
            return [item["name"] for item in items if isinstance(item, dict) and "name" in item]
        return []
    except:
        return []

movies["genres_list"] = movies["genres"].fillna("[]").apply(parse_names)
movies["keywords_list"] = movies["keywords"].fillna("[]").apply(parse_names)

movies[["title", "genres_list", "keywords_list"]].head()

## 4. Creación de la sopa de contenido

La “sopa” combina palabras clave y géneros en una sola cadena de texto.  
Esta cadena se usará para calcular similitud entre películas.

In [ ]:
def clean_text_list(items):
    return [str(item).lower().replace(" ", "") for item in items]

movies["genres_clean"] = movies["genres_list"].apply(clean_text_list)
movies["keywords_clean"] = movies["keywords_list"].apply(clean_text_list)

movies["soup"] = movies["genres_clean"].apply(lambda x: " ".join(x)) + " " + movies["keywords_clean"].apply(lambda x: " ".join(x))

movies[["title", "soup"]].head()

## 5. Selección de datos para el recomendador

Para que el cálculo sea más rápido, se eliminan películas sin título o sin información útil.  
También se reinicia el índice para trabajar correctamente con la matriz de similitud.

In [ ]:
rec_movies = movies[
    movies["title"].notna() &
    movies["soup"].str.strip().ne("")
].copy()

rec_movies = rec_movies.reset_index(drop=True)

print("Películas disponibles para recomendación:", rec_movies.shape[0])
rec_movies[["title", "soup"]].head()

## 6. Vectorización y similitud coseno

CountVectorizer transforma la sopa de palabras en una matriz numérica.  
Después se calcula la similitud coseno para medir qué tan parecidas son las películas entre sí.

In [ ]:
count = CountVectorizer(stop_words="english")
count_matrix = count.fit_transform(rec_movies["soup"])

cosine_sim = cosine_similarity(count_matrix, count_matrix)

indices = pd.Series(rec_movies.index, index=rec_movies["title"]).drop_duplicates()

print("Matriz de similitud creada:", cosine_sim.shape)

## 7. Recomendador básico por similitud

Esta función recibe el título de una película y devuelve las películas más similares según géneros y palabras clave.

In [ ]:
def recomendar_peliculas(titulo, num_recomendaciones=10):
    if titulo not in indices:
        return f"No se encontró la película: {titulo}"

    idx = indices[titulo]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:num_recomendaciones + 1]
    movie_indices = [i[0] for i in sim_scores]

    return rec_movies[["title", "vote_average", "vote_count"]].iloc[movie_indices]

recomendar_peliculas("Toy Story", 10)

## 8. Mejora: filtro de popularidad

Una limitación del recomendador básico es que puede sugerir películas muy similares, pero poco confiables por tener pocos votos.  
Para resolver esto, se agrega una calificación ponderada basada en la fórmula de IMDb.

In [ ]:
C = rec_movies["vote_average"].mean()
m = rec_movies["vote_count"].quantile(0.90)

print("Promedio general de calificación:", round(C, 2))
print("Mínimo de votos requerido:", round(m, 2))

def weighted_rating(x, m=m, C=C):
    v = x["vote_count"]
    R = x["vote_average"]
    return (v / (v + m) * R) + (m / (m + v) * C)

rec_movies["score"] = rec_movies.apply(weighted_rating, axis=1)

rec_movies[["title", "vote_average", "vote_count", "score"]].head()

## 9. Recomendador mejorado

Esta versión toma las 30 películas más similares, calcula su calificación ponderada y devuelve las 10 mejores.  
Con esto se combinan similitud y popularidad.

In [ ]:
def recomendar_peliculas_mejorado(titulo, num_similares=30, num_recomendaciones=10):
    if titulo not in indices:
        return f"No se encontró la película: {titulo}"

    idx = indices[titulo]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:num_similares + 1]
    movie_indices = [i[0] for i in sim_scores]

    similares = rec_movies.iloc[movie_indices].copy()
    similares = similares.sort_values("score", ascending=False)

    return similares[["title", "genres_list", "vote_average", "vote_count", "score"]].head(num_recomendaciones)

recomendar_peliculas_mejorado("Toy Story")

## 10. Top de películas populares

También se puede crear un recomendador general de películas populares usando únicamente la calificación ponderada.

In [ ]:
top_populares = rec_movies[
    rec_movies["vote_count"] >= m
].sort_values("score", ascending=False)

top_populares[["title", "vote_average", "vote_count", "score"]].head(10)

## 11. Ejemplos con otras películas

In [ ]:
recomendar_peliculas_mejorado("The Dark Knight")

In [ ]:
recomendar_peliculas_mejorado("Avatar")

## Conclusión

El sistema recomendador permite encontrar películas similares usando información de contenido como géneros y palabras clave.  
La mejora con calificación ponderada ayuda a ordenar las recomendaciones considerando también la popularidad y la confiabilidad de las calificaciones.

Esto permite obtener recomendaciones más útiles, ya que no solo se consideran películas parecidas, sino también películas con suficiente respaldo de votos.